# 🤖 ReAct Agent: Designing Autonomous Cognitive Loops (JARVIS Phase 1)
### *A Blueprint for LLM-based Tool Integration, Reasoning, and Self-Correction*

In Phase 1 of our JARVIS roadmap, we transition from static mathematical updates (like Q-learning or DQN) to a **cognitive reasoning model**. Instead of outputting simple action indices, the agent uses a Large Language Model (LLM) to perform **ReAct** (Reasoning + Action) loops.

---

## 🌐 Section 1: Conceptual Architecture
The **ReAct** paradigm (Yao et al., 2022) combines reasoning and acting in LLMs. The agent generates reasoning traces ("Thoughts") and task-specific actions ("Actions"), which return environment responses ("Observations").

```
   +-------------------------------------------------------------+
   |                                                             |
   |                       User Request                          |
   |                                                             |
   +------------------------------+------------------------------+
                                  | 
                                  v 
   +------------------------------+------------------------------+
   |               1. THOUGHT (Reasoning Step)                   |
   |  "What tool should I use to solve this next?"               |
   +------------------------------+------------------------------+
                                  | 
                                  v 
   +------------------------------+------------------------------+
   |                 2. ACTION (Tool Call)                       |
   |  Agent executes: ToolName[argument]                         |
   +------------------------------+------------------------------+
                                  | 
                                  v 
   +------------------------------+------------------------------+
   |               3. OBSERVATION (Tool Output)                  |
   |  System returns error messages or computed outputs          |
   +------------------------------+------------------------------+
                                  | 
                                  +---> Loop repeats until
                                        Final Answer is reached
```

### Why this is JARVIS-like:
- **Dynamic tool usage**: The model decides when to execute code, search for information, or run computations.
- **Self-correction**: If the Python code it executes fails with a traceback, the agent observes the traceback, understands its syntax error, and corrects its code in the next iteration.

## 🛠️ Section 2: Setup
We will install the Google Generative AI library (`google-generativeai`) to connect with Google's Gemini models. We also import standard Python libraries for code execution monitoring.

In [ ]:
# Install Google Generative AI client (required for real LLM reasoning)
!pip install google-generativeai -q

import os
import sys
import re
import random
from io import StringIO
import google.generativeai as genai

print("Setup completed successfully!")

## ⚙️ Section 3: Tool Definitions
Here, we implement the action space for the agent. We define:
1. **`PythonREPL[code]`**: Executes arbitrary Python code safely, capturing stdout/stderr and returning it to the agent.
2. **`Calculate[expr]`**: Evaluates basic math expressions using Python's mathematical parsing.
3. **`WebSearch[query]`**: A simulated web search tool to demonstrate retrieval integrations.

In [ ]:
def execute_python_code(code: str) -> str:
    """Executes a string of Python code and returns stdout and stderr."""
    old_stdout = sys.stdout
    old_stderr = sys.stderr
    sys.stdout = mystdout = StringIO()
    sys.stderr = mystderr = StringIO()
    try:
        exec(code, globals())
        output = mystdout.getvalue()
        error = mystderr.getvalue()
    except Exception as e:
        output = mystdout.getvalue()
        error = str(e)
    finally:
        sys.stdout = old_stdout
        sys.stderr = old_stderr
    
    if error:
        return f"Execution Error:\n{error}"
    return output.strip() if output.strip() else "Executed successfully (no stdout output)."

def calculate_expression(expression: str) -> str:
    """Evaluates mathematical expressions safely."""
    try:
        # Simple cleaning of input bounds
        clean_expr = re.sub(r'[^0-9+\-*/().\s]', '', expression)
        result = eval(clean_expr)
        return str(result)
    except Exception as e:
        return f"Calculation Error: {str(e)}"

def simulated_search(query: str) -> str:
    """A mock search engine that returns predefined results for demonstration."""
    database = {
        "jarvis project overview": "JARVIS is a cognitive self-improving assistant focused on automating operating system actions and code generation.",
        "reinforcement learning agent design": "RL agents optimize behavior by maximizing cumulative rewards using exploration-exploitation strategies.",
        "latest stable baselines3": "Stable-Baselines3 version 2.8 contains implementations of PPO, DQN, SAC, and DDPG built on PyTorch."
    }
    query_clean = query.lower().strip()
    for key in database:
        if key in query_clean or query_clean in key:
            return database[key]
    return f"Search returned 0 results for: '{query}'"

## 🧠 Section 4: The Agent Controller & System Prompt
To direct the agent to think and act, we feed it a **System Prompt** defining the format guidelines. We also build a **Mock Agent fallback** so that the notebook can be tested fully without requiring an API key immediately.

In [ ]:
SYSTEM_PROMPT = """
You are an autonomous cognitive agent (JARVIS Phase 1).
Solve the user's task using the following structure for each step:

Thought: your reasoning about what to do next.
Action: the tool to use, formatted exactly as ToolName[argument].
Observation: the result of the tool execution (provided to you).

Available Tools:
1. PythonREPL[code]: Executes python code and returns its printed output.
2. Calculate[expression]: Computes basic mathematical expressions.
3. WebSearch[query]: Queries a simulated knowledge database.

Once you have gathered enough information to answer, output:
Final Answer: your final solution.

Example Trace:
Task: Calculate 25 * 43.
Thought: I need to multiply 25 and 43. I will use the Calculate tool.
Action: Calculate[25 * 43]
Observation: 1075
Thought: The calculation is complete. I can now provide the final answer.
Final Answer: 1075
"""

# Fallback simulator class representing the LLM behavior for testing out-of-the-box
class MockGeminiModel:
    def __init__(self):
        self.step = 0
        
    def generate_content(self, prompt: str):
        # Simulates the step-by-step reasoning trace of an agent solving the Fibonacci sequence
        if "15th Fibonacci" in prompt:
            if self.step == 0:
                self.step += 1
                return MockResponse(
                    "Thought: I need to write a python script to compute the 15th Fibonacci number and print it. I will use the PythonREPL tool.\n"
                    "Action: PythonREPL[def fib(n):\n    if n <= 1: return n\n    return fib(n-1) + fib(n-2)\nprint(fib(15))]"
                )
            elif "610" in prompt:
                return MockResponse(
                    "Thought: The PythonREPL executed successfully and returned 610. I can now present the final answer.\n"
                    "Final Answer: The 15th Fibonacci number is 610."
                )
        return MockResponse(
            "Thought: I don't recognize this task in mock mode. Please configure a valid API key to run custom tasks.\n"
            "Final Answer: Error - Gemini API key required."
        )

class MockResponse:
    def __init__(self, text):
        self.text = text

## 🔄 Section 5: The ReAct Loop Execution Engine
We write the orchestrator loop that manages the execution. It parses the LLM output, extracts the `Action` call, triggers the corresponding tool function, and appends the `Observation` to the conversation history before prompting the LLM again.

In [ ]:
class ReActAgent:
    def __init__(self, api_key: str = None):
        if api_key:
            # Configure actual Gemini API client if key is provided
            genai.configure(api_key=api_key)
            self.model = genai.GenerativeModel('gemini-1.5-flash')
            self.real_api = True
        else:
            # Fallback to simulated controller
            self.model = MockGeminiModel()
            self.real_api = False
            
    def run(self, task: str, max_iterations: int = 5):
        # Initialize conversation log with task and system instructions
        history = f"{SYSTEM_PROMPT}\nTask: {task}\n"
        print(f"🚀 Starting Agent Execution for Task: '{task}'")
        print("="*60)
        
        for i in range(max_iterations):
            print(f"\n👉 Iteration {i+1}...")
            
            # Query LLM
            response = self.model.generate_content(history)
            llm_text = response.text.strip()
            print(llm_text)
            
            # Append LLM reasoning to memory log
            history += f"{llm_text}\n"
            
            # Check if final answer is outputted
            if "Final Answer:" in llm_text:
                print("\n✅ Task Completed!")
                return
                
            # Parse Action
            match = re.search(r'Action:\s*(\w+)\[(.*)\]', llm_text, re.DOTALL)
            if not match:
                print("\n⚠️ Formatting error - no action parsed. Exiting loop.")
                break
                
            tool_name = match.group(1)
            tool_arg = match.group(2)
            
            # Execute Tools
            print(f"\n🔧 Executing Tool: {tool_name} with arg...")
            if tool_name == "PythonREPL":
                observation = execute_python_code(tool_arg)
            elif tool_name == "Calculate":
                observation = calculate_expression(tool_arg)
            elif tool_name == "WebSearch":
                observation = simulated_search(tool_arg)
            else:
                observation = f"Tool {tool_name} is not recognized."
                
            print(f"📡 Observation: {observation}")
            
            # Feed tool outcome back to the history
            history += f"Observation: {observation}\n"
            
        print("\n❌ Max iterations reached without a Final Answer.")

## 🧪 Section 6: Testing the Agent
We now initialize the agent and execute a demo run to find the 15th Fibonacci number. 

*Note: You can supply an active Gemini API key inside the constructor `ReActAgent(api_key="YOUR_API_KEY")` to test any open-ended task.*

In [ ]:
# To use actual LLM reasoning, replace None with your Google AI Studio API key
GEMINI_API_KEY = None 

agent = ReActAgent(api_key=GEMINI_API_KEY)
agent.run("Calculate the 15th Fibonacci number.")